In [1]:
import numpy as np
import pandas as pd
import os 
import sys
import pyomo.environ as pyo


# module_path = os.path.abspath(os.path.join('../'))

# if module_path not in sys.path:
#     sys.path.append(module_path)

# # Define time intervals
# time_intervals = range(48)  # Example: 48 half-hour intervals in a day

# # Define Parameters
# battery_capacity = 100  # Example: 100 kWh
# initial_charge = 50 / battery_capacity  # Normalized initial charge
# charge_efficiency = 0.95
# discharge_efficiency = 0.95
# max_charge_rate = 10 / 2  # kW per half-hour
# max_discharge_rate = 10 / 2  # kW per half-hour
# electricity_price = np.random.rand(48)  # Example: random prices for 48 half-hours

# # Create a Pyomo model
# model = pyo.ConcreteModel()

# # Define Variables
# model.charge = pyo.Var(time_intervals, domain=pyo.NonNegativeReals)
# model.discharge = pyo.Var(time_intervals, domain=pyo.NonNegativeReals)
# model.state_of_charge = pyo.Var(time_intervals, domain=pyo.NonNegativeReals, bounds=(0, 1))  # Normalized SOC

# # Define Objective Function
# def objective_rule(model):
#     return sum(electricity_price[t] * (model.charge[t] - model.discharge[t]) for t in time_intervals)
# model.objective = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# # Define Constraints
# # Initial state of charge
# model.initial_soc = pyo.Constraint(expr=model.state_of_charge[0] == initial_charge)

# # Charge and discharge rate constraints
# model.charge_rate_constraint = pyo.ConstraintList()
# model.discharge_rate_constraint = pyo.ConstraintList()
# for t in time_intervals:
#     model.charge_rate_constraint.add(model.charge[t] <= max_charge_rate)
#     model.discharge_rate_constraint.add(model.discharge[t] <= max_discharge_rate)

# # State of charge update
# model.soc_update = pyo.ConstraintList()
# for t in time_intervals:
#     if t == 0:
#         continue
#     model.soc_update.add(model.state_of_charge[t] == model.state_of_charge[t-1] + (charge_efficiency * model.charge[t-1] - model.discharge[t-1] / discharge_efficiency) / battery_capacity)

# # Solve the model
# solver = pyo.SolverFactory('glpk')
# solver.solve(model)

# # Display the results
# for t in time_intervals:
#     print(f"Half-hour {t}: Charge = {model.charge[t]()} kW, Discharge = {model.discharge[t]()} kW, State of Charge = {model.state_of_charge[t]()} ")


In [22]:
model = pyo.ConcreteModel()
solver = pyo.SolverFactory('glpk')


model.x = pyo.Var([1,2], domain=pyo.NonNegativeReals)
model.OBJ = pyo.Objective(expr = 2*model.x[1] + 3*model.x[2],sense=pyo.minimize)
model.constraints_1 = pyo.Constraint(expr = 3*model.x[1] + 4*model.x[2] >= 1)

In [27]:
solver.solve(model)

{'Problem': [{'Name': 'unknown', 'Lower bound': 0.666666666666667, 'Upper bound': 0.666666666666667, 'Number of objectives': 1, 'Number of constraints': 1, 'Number of variables': 2, 'Number of nonzeros': 2, 'Sense': 'minimize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Statistics': {'Branch and bound': {'Number of bounded subproblems': 0, 'Number of created subproblems': 0}}, 'Error rc': 0, 'Time': 0.0309450626373291}], 'Solution': [OrderedDict([('number of solutions', 0), ('number of solutions displayed', 0)])]}

In [28]:
model.x[1].value, model.x[2].value

(0.333333333333333, 0.0)

In [17]:
def X_init(m):
    for i in range(10):
        yield 2*i+1
model.X = pyo.Set(initialize=X_init)



In [31]:
model.pprint()

1 Var Declarations
    x : Size=2, Index={1, 2}
        Key : Lower : Value             : Upper : Fixed : Stale : Domain
          1 :     0 : 0.333333333333333 :  None : False : False : NonNegativeReals
          2 :     0 :               0.0 :  None : False : False : NonNegativeReals

1 Objective Declarations
    OBJ : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 2*x[1] + 3*x[2]

1 Constraint Declarations
    constraints_1 : Size=1, Index=None, Active=True
        Key  : Lower : Body            : Upper : Active
        None :   1.0 : 3*x[1] + 4*x[2] :  +Inf :   True

3 Declarations: x OBJ constraints_1


In [32]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
model = pyo.ConcreteModel()
model.nVars = pyo.Param(initialize=4)
model.N = pyo.RangeSet(model.nVars)
model.x = pyo.Var(model.N, within=pyo.Binary)
model.obj = pyo.Objective(expr=pyo.summation(model.x))
model.cuts = pyo.ConstraintList()
opt = SolverFactory('glpk')
opt.solve(model) 

# Iterate, adding a cut to exclude the previously found solution
for i in range(5):
   expr = 0
   for j in model.x:
       if pyo.value(model.x[j]) < 0.5:
           expr += model.x[j]
       else:
           expr += (1 - model.x[j])
   model.cuts.add( expr >= 1 )
   results = opt.solve(model)
   print ("\n===== iteration",i)
   model.display() 


===== iteration 0
Model unknown

  Variables:
    x : Size=4, Index=N
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :   1.0 :     1 : False : False : Binary
          2 :     0 :   0.0 :     1 : False : False : Binary
          3 :     0 :   0.0 :     1 : False : False : Binary
          4 :     0 :   0.0 :     1 : False : False : Binary

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True :   1.0

  Constraints:
    cuts : Size=1
        Key : Lower : Body : Upper
          1 :   1.0 :  1.0 :  None

===== iteration 1
Model unknown

  Variables:
    x : Size=4, Index=N
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :   0.0 :     1 : False : False : Binary
          2 :     0 :   1.0 :     1 : False : False : Binary
          3 :     0 :   0.0 :     1 : False : False : Binary
          4 :     0 :   0.0 :     1 : False : False : Binary

  Objectives:
    obj : Si

In [38]:
import pyomo.environ as pyo
A = ['hammer', 'wrench', 'screwdriver', 'towel']
b = {'hammer':8, 'wrench':3, 'screwdriver':6, 'towel':11}
w = {'hammer':5, 'wrench':7, 'screwdriver':4, 'towel':3}

In [39]:
W_max = 14
model = pyo.ConcreteModel()
model.x = pyo.Var( A, within=pyo.Binary )

In [40]:
model.value = pyo.Objective(expr = sum( b[i]*model.x[i] for i in A ), sense = pyo.maximize )
model.weight = pyo.Constraint(expr = sum( w[i]*model.x[i] for i in A) <= W_max )

In [41]:
opt = pyo.SolverFactory('glpk')
result_obj = opt.solve(model, tee=True)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmp_hu2chl1.glpk.raw
 --wglp /var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpia8p7i87.glpk.glp
 --cpxlp /var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpso9ow1z1.pyomo.lp
Reading problem data from '/var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpso9ow1z1.pyomo.lp'...
/var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpso9ow1z1.pyomo.lp:25: warning: lower bound of variable 'x2' redefined
/var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpso9ow1z1.pyomo.lp:25: warning: upper bound of variable 'x2' redefined
1 row, 4 columns, 4 non-zeros
4 integer variables, all of which are binary
29 lines were read
Writing problem data to '/var/folders/pq/0b742hwn67n8b50cr4g_qz1w0000gr/T/tmpia8p7i87.glpk.glp'...
17 lines were written
GLPK Integer Optimizer 5.0
1 row, 4 columns, 4 non-zeros
4 integer variables, all of which are binary
Preprocessing...
1 constrai

In [42]:
model.pprint()

1 Var Declarations
    x : Size=4, Index={hammer, wrench, screwdriver, towel}
        Key         : Lower : Value : Upper : Fixed : Stale : Domain
             hammer :     0 :   1.0 :     1 : False : False : Binary
        screwdriver :     0 :   1.0 :     1 : False : False : Binary
              towel :     0 :   1.0 :     1 : False : False : Binary
             wrench :     0 :   0.0 :     1 : False : False : Binary

1 Objective Declarations
    value : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 8*x[hammer] + 3*x[wrench] + 6*x[screwdriver] + 11*x[towel]

1 Constraint Declarations
    weight : Size=1, Index=None, Active=True
        Key  : Lower : Body                                                      : Upper : Active
        None :  -Inf : 5*x[hammer] + 7*x[wrench] + 4*x[screwdriver] + 3*x[towel] :  14.0 :   True

3 Declarations: x value weight


In [43]:
model.display()

Model unknown

  Variables:
    x : Size=4, Index={hammer, wrench, screwdriver, towel}
        Key         : Lower : Value : Upper : Fixed : Stale : Domain
             hammer :     0 :   1.0 :     1 : False : False : Binary
        screwdriver :     0 :   1.0 :     1 : False : False : Binary
              towel :     0 :   1.0 :     1 : False : False : Binary
             wrench :     0 :   0.0 :     1 : False : False : Binary

  Objectives:
    value : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True :  25.0

  Constraints:
    weight : Size=1
        Key  : Lower : Body : Upper
        None :  None : 12.0 :  14.0


In [61]:
import pyomo.environ as pyo
model = pyo.ConcreteModel(name="(WL)")
W = ['Harlingen', 'Memphis', 'Ashland']
C = ['NYC', 'LA', 'Chicago', 'Houston']
d = {('Harlingen', 'NYC'): 1956, ('Memphis', 'NYC'): 1096, ('Ashland', 'NYC'): 610,
     ('Harlingen', 'LA'): 1606, ('Memphis', 'LA'): 1792, ('Ashland', 'LA'): 2065,
     ('Harlingen', 'Chicago'): 1410, ('Memphis', 'Chicago'): 531, ('Ashland', 'Chicago'): 324,
     ('Harlingen', 'Houston'): 330, ('Memphis', 'Houston'): 1745, ('Ashland', 'Houston'): 1236}
P = 2

In [62]:
model.x = pyo.Var(W, C, bounds=(0,1))
model.y = pyo.Var(W, within=pyo.Binary)

In [63]:
@model.Objective()
def obj(m):
    return sum(d[w,c]*m.x[w,c] for w in W for c in C)

In [64]:
@model.Constraint(C)
def one_per_cust(m, c):
    return sum(m.x[w,c] for w in W) == 1

In [65]:
@model.Constraint(W, C)
def warehouse_active (m, w, c):
    return m.x[w,c] <= m.y[w]

In [66]:
@model.Constraint()
def num_warehouses (m):
    return sum(m.y[w] for w in W) <= P

In [67]:
pyo.SolverFactory('glpk').solve(model)

{'Problem': [{'Name': 'unknown', 'Lower bound': 2870.0, 'Upper bound': 2870.0, 'Number of objectives': 1, 'Number of constraints': 17, 'Number of variables': 15, 'Number of nonzeros': 39, 'Sense': 'minimize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Statistics': {'Branch and bound': {'Number of bounded subproblems': '1', 'Number of created subproblems': '1'}}, 'Error rc': 0, 'Time': 0.054451942443847656}], 'Solution': [OrderedDict([('number of solutions', 0), ('number of solutions displayed', 0)])]}

In [68]:
model.display()

Model '(WL)'

  Variables:
    x : Size=12, Index={Harlingen, Memphis, Ashland}*{NYC, LA, Chicago, Houston}
        Key                      : Lower : Value : Upper : Fixed : Stale : Domain
          ('Ashland', 'Chicago') :     0 :   1.0 :     1 : False : False :  Reals
          ('Ashland', 'Houston') :     0 :   0.0 :     1 : False : False :  Reals
               ('Ashland', 'LA') :     0 :   0.0 :     1 : False : False :  Reals
              ('Ashland', 'NYC') :     0 :   1.0 :     1 : False : False :  Reals
        ('Harlingen', 'Chicago') :     0 :   0.0 :     1 : False : False :  Reals
        ('Harlingen', 'Houston') :     0 :   1.0 :     1 : False : False :  Reals
             ('Harlingen', 'LA') :     0 :   1.0 :     1 : False : False :  Reals
            ('Harlingen', 'NYC') :     0 :   0.0 :     1 : False : False :  Reals
          ('Memphis', 'Chicago') :     0 :   0.0 :     1 : False : False :  Reals
          ('Memphis', 'Houston') :     0 :   0.0 :     1 : False : False